----
## <font color='CornflowerBlue'>Practical 3: Neural networks with graphs and sequences</font> 
##### Alok Bharadwaj and Arjen Jakobi
---

## 1.1 Recognising graph-structured data and defining graphs

Last week, you had a short introduction to graph-structured data. Here, we will recap the main ideas with a short exercise. 

The goal is to practise identifying **nodes** and **edges** in a graph. You will construct a graph based on the following text, adapted from *Murder on the Orient Express* by Agatha Christie.

>> The Orient Express had stalled in the night, snowbound between Belgrade and Brod. In the dining car, the famous Belgian detective Hercule Poirot took his coffee in silence.
Across from him sat M. Bouc, an old friend and director of the Compagnie Internationale des Wagons-Lits. "A curious assortment, mon ami," Bouc remarked. "All nations, all classes — and all snowbound together."
The conductor, Pierre Michel, greeted Mr Poirot and moved between tables with efficiency. "Pierre has been with us fifteen years," Bouc said as he passed. "A good man."
Poirot's gaze travelled the carriage. There was the wealthy American, Mr. Ratchett, with his hard, narrow eyes who had insisted to M. Bouc that he be seated in the carriage next to Mr Poirot. Beside him sat his secretary, Hector MacQueen, employed only six months. Earlier, Poirot had seen MacQueen speaking easily with Colonel Arbuthnot, a retired British Officer.
The Colonel had befriended a Miss Mary Debenham, a young Englishwoman, on the leg from Aleppo. Poirot had overheard them once in a darkened corridor — "Not now," she had said. "When it's all over. When it's behind us."
Further down, Princess Dragomiroff held court with imperial dignity. Her maid, Hildegarde Schmidt, had attended her for thirty years. The Princess nodded to the diplomatic pair across the aisle — Count and Countess Andrenyi of Hungary — old social acquaintances and asked Pierre Michel to bring them some wine. The Countess was a former opera singer, and the Count a diplomat.
After dinner, Ratchett rose and approached Poirot.
"Mr. Poirot. I have enemies, sir. I should like to engage your services."
Poirot regarded him a long moment. "I regret, monsieur, that I do not accept new clients."

Construct a graph based on the text fragemnt above. In this graph:

- **Nodes** represent the characters mentioned in the text.
- **Edges** represent whether two characters know each other.

Read the text carefully and decide which characters should be connected. As you construct the graph, think about the following questions:

- Which relationships are clearly stated?
- Are there any ambiguous cases?
- Are some relationships one-sided (directed)?
- Does “knowing about someone” count as “knowing them personally”?
- Should all relationships be treated equally, or should some edges have different strengths?

There may not be one single "correct" graph. The aim is to practise interpreting text as graph-structured data and to discuss the choices involved in defining nodes and edges. Let's first define all `nodes`. Check if the following list is complete.

In [ ]:
import networkx as nx

G = nx.Graph()
G.add_node(1, name="Hercule Poirot")
G.add_node(2, name="M. Bouc")
# Add more nodes for other characters
G.add_node(3, name="Pierre")
G.add_node(4, name="Ratchett")
G.add_node(5, name="MacQueen")
G.add_node(6, name="Arbuthnot")
G.add_node(7, name="Mary")
G.add_node(8, name="Dragomiroff")
G.add_node(9, name="Hildegarde")
G.add_node(10, name="Count Andrenyi")
G.add_node(11, name="Countess Andrenyi")


Now define the `edges` by connecting the relevant nodes. For the moment we will consider all edges to have equal weight/strength.

In [ ]:
# Who knew who in the story
G.add_edge(1, 2)  # Hercule Poirot knew M. Bouc
# Add more edges to represent relationships between characters
G.add_edge(1, 3)  # Hercule Poirot greets Pierre
G.add_edge(2, 3)  # M. Bouc hired Pierre 
G.add_edge(4, 5) # Ratchett hired MacQueen
G.add_edge(5, 6) # MacQueen knew Arbuthnot
G.add_edge(6, 7) # Arbuthnot knew Mary
G.add_edge(8, 9) # Dragomiroff knew Hildegarde
G.add_edge(10, 11) # Count Andrenyi knew Countess Andrenyi
G.add_edge(8, 10) # Dragomiroff knew Count Andrenyi
G.add_edge(8, 11) # Dragomiroff knew Countess Andrenyi
G.add_edge(1, 4) # Hercule Poirot knew Ratchett
#G.add_edge(1, 6) # Poirot overheard Arbuthnot talking to Mary 
#G.add_edge(1, 7) # Poirot overheard Arbuthnot talking to Mary
G.add_edge(3, 8) 
G.add_edge(4, 2) 


Let's have a look at your graph!

In [ ]:
layout = nx.spring_layout(G, seed=1, k=0.75)  # Adjust k for better spacing
node_labels = {node: data['name'] for node, data in G.nodes(data=True)}
nx.draw(G, pos=layout, labels=node_labels, with_labels=True, node_size=800, node_color='lightblue', font_size=8)

Upload your graph:

In [ ]:
from src.utils import upload_to_surfdrive 
upload_to_surfdrive(graph=G) 

Curious about how detective stories can be turned into graphs? Check out this blog post: [Dead Men Tell No Tales — But Graphs Do](https://medium.com/@alexzhai/dead-men-tell-no-tales-but-graphs-do-005601a1cdf9).

## 1.2 Protein structures as graphs

### Data models for representation of protein structure
Before a protein structure can be converted into a graph, a point cloud, or a tensor, information on the arrangement of atoms in the protein structure first needs to be parsed into a reliable data structure. This also means that there should be some convention in which such data is intially archived. Protein structures are commonly stored in text-based coordinate files. We will briefly look at two widely used formats: **PDB** and **mmCIF**. Both formats describe the same kind of information: atoms, residues, chains, coordinates, experimental metadata, and sometimes biological assemblies or ligands. However, they organise this information in different ways.

We will use **1L2Y** as an example structure and download it in both PDB and mmCIF format.

In [ ]:
import os 
# Downloading the data for the GNN class if you are not using DelftBlue. Uncomment the next two lines
#!git clone https://github.com/courtel/GNN_dl_workshop.git gnn_data
#!wget https://files.rcsb.org/download/1L2Y.pdb gnn_data/1L2Y.pdb

# Point this to the appropriate gnn_data folder
GNN_DATA_FOLDER = "/projects/nb4170/gnn_data" 

pdb_file = os.path.join(GNN_DATA_FOLDER, "1L2Y.pdb")
cif_file = os.path.join(GNN_DATA_FOLDER, "1L2Y.cif")

#### How PDB files are organised
---

Let us have a look at the content of an PDB file, which is still the most widely used data format for macromolecular coordinates. The PDB format makes the coordinate records very visible: each atom appears on one line. However, the format is also quite old, and many fields are encoded by position rather than by explicit names. This means that we (or a file parser) needs to know exactly which character columns correspond to atom names, residue names, chain identifiers, and coordinates.


In [ ]:
from src.utils import show_pdb_atom_records
show_pdb_atom_records(pdb_file, n=10)

PDB coordinate files are organised as fixed-width records.
Each `ATOM` or `HETATM` line stores one atom.
The position of the text in the line determines the meaning of the value.

```
  1-6    record type, usually ATOM or HETATM
  7-11   atom serial number
  13-16  atom name
  18-20  residue name
  22     chain identifier
  23-26  residue sequence number
  31-38  x coordinate
  39-46  y coordinate
  47-54  z coordinate
  55-60  occupancy
  61-66  B-factor / temperature factor
  77-78  chemical element
  ```


#### How mmCIF files are organised
---
The mmCIF format stores information in a more explicit and table-like way. An mmCIF file contains one or more `data_` blocks. Inside each block, information is stored as:

- individual tag-value pairs
- repeated tables introduced by `loop_`

The mmCIF format was introduced to overcome several limitations of the older PDB format. It is more verbose, but also more explicit. Instead of relying on fixed character positions, mmCIF stores data using named fields. This makes it easier to represent large structures and richer metadata.

The most important coordinate table for our purposes is usually `_atom_site`. This table contains one row per atom and named columns for the atom name, residue name, chain, residue number, coordinates, occupancy, and B-factor.

In [ ]:
from src.utils import show_mmcif_atom_site_loop
show_mmcif_atom_site_loop(cif_file, n=10)

The `_atom_site` category plays a similar role to the `ATOM` and `HETATM` records in a PDB file, but it is organised as a table with named columns.

```
  _atom_site.label_atom_id      atom name
  _atom_site.label_comp_id      residue name
  _atom_site.auth_asym_id       chain identifier
  _atom_site.auth_seq_id        residue number
  _atom_site.Cartn_x            x coordinate
  _atom_site.Cartn_y            y coordinate
  _atom_site.Cartn_z            z coordinate
  _atom_site.occupancy          occupancy
  _atom_site.B_iso_or_equiv     B-factor
```

#### Representing structural hierarchies

We typically access data structures such as PDB or mmcif files through file parsers. Parsing means converting the raw text file into structured Python objects. A parser such as [`gemmi`](https://gemmi.readthedocs.io/en/latest/) reads the PDB or mmCIF file, interprets the records or tables, and builds a hierarchy that can be used programmatically.

For example, instead of manually slicing columns from a PDB line or searching for `_atom_site.Cartn_x` in an mmCIF loop, we can ask `gemmi` for:

```text
Structure
  Model
    Chain
      Residue
        Atom
```

This is important because the raw PDB and mmCIF files look very different, but after parsing they can be accessed through the same conceptual structure. This allows us to write one downstream analysis workflow that works for both file formats.


In [ ]:
from src.utils import show_gemmi_structure_summary, show_gemmi_atom_table
show_gemmi_structure_summary(pdb_file)
show_gemmi_structure_summary(cif_file)
show_gemmi_atom_table(cif_file, n=10)

Once the file has been parsed, we can easily compute useful properties of the protein, such as:

- the number of chains
- the number of residues
- the number of atoms
- the amino-acid sequence
- the average B-factor

These properties are useful for both structural biology and machine learning. For example, a protein graph might use residues as nodes, distances between residues as edges, and properties such as residue identity, coordinates, or B-factors as features. 

In [ ]:
def calculate_mean_b_factor(file_path):
    """
    Calculate the mean isotropic B-factor for all atoms in a PDB or mmCIF file.
    """
    import gemmi
    import numpy as np

    structure = gemmi.read_structure(str(file_path), merge_chain_parts=False)

    b_factors = [
        atom.b_iso
        for model in structure
        for chain in model
        for residue in chain
        for atom in residue
    ]

    if len(b_factors) == 0:
        return None

    return float(np.mean(b_factors))

mean_b = calculate_mean_b_factor(pdb_file)
print(f"Mean B-factor: {mean_b:.2f} Å^2")

### Representing Proteins as Graphs

Proteins can be represented as graphs at different levels of detail. The choice of representation depends on the question we want to answer.

In a protein graph:

- **Nodes** represent parts of the protein.
- **Edges** represent relationships between those parts.
- **Node and edge features** store useful information, such as residue type, atom type, distance, or interaction type.

#### Atomic-level graphs

**Nodes:** atoms  
**Edges:** chemical bonds or spatial contacts  

Atomic graphs keep detailed chemical information and are useful for tasks such as ligand binding, active-site modelling, or protein–ligand interactions.

#### Residue-level graphs

**Nodes:** amino acid residues  
**Edges:** peptide bonds or spatial contacts between residues  

Residue graphs are commonly used because they capture the overall 3D structure while keeping the graph smaller than an atom-level graph. They are useful for tasks such as function prediction, fold classification, mutation effect prediction, and binding-site prediction.

#### Domain-level graphs

**Nodes:** domains or larger structural regions  
**Edges:** contacts or relationships between domains  

Domain graphs are useful for studying large proteins, protein complexes, conformational change, or long-range communication.

In this exercise, we will mainly use a **residue-level graph**, where each node is an amino acid residue and edges describe structural relationships between residues.

A protein structure can be represented as a graph by choosing what the **nodes** and **edges** should mean.

### Choosing the right level of abstraction

The graph representation determines what information the model can use.

A very detailed graph, such as an atomic graph, contains rich chemical information but can be large and computationally expensive. A coarser graph, such as a residue-level or domain-level graph, is smaller and easier to analyse, but some fine-grained information is lost.

The key question is:

> What information is important for the task?

For example, if we want to model ligand binding, atom-level detail may be important. If we want to classify protein folds, a residue-level graph may be sufficient. If we want to study large-scale domain motion, a domain-level graph may be more appropriate.

In this exercise, we will mainly use a **residue-level graph**, where each node represents an amino acid residue and edges represent structural relationships between residues.

In [ ]:
import os
import graphein.protein as gp
from graphein.protein.graphs import construct_graph

params_1 = {
    "granularity": "CA",
    "edge_construction_functions": [
        gp.add_peptide_bonds,
    ],
}

config_1 = gp.ProteinGraphConfig(**params_1)

protein_graph_1 = construct_graph(config=config_1, path=os.path.join(GNN_DATA_FOLDER, "1L2Y.pdb"))

In [ ]:
from src.utils import display_protein_with_graph
display_protein_with_graph(protein_graph_1)

#### __Some questions for you:__

#### 1. Propose a method to construct a feature_vector for each node in the graph. What features would you include? How many dimensions would your feature vector have?

In [ ]:
Answer: 

#### 2. What are the properties of the adjacency matrix for this protein? 

In [ ]:
Answer: 

#### 3. How would the graph change if we added a ligand that binds to the protein?

In [ ]:
Answer:

----
#### Pause here (and get some water, coffee or tea).

Wait until everyone has completed the exercise.

Then we will move on to the next module, where we will learn the message passing algorithm together as a group. This part will require collaboration with your neighbours.